# Script 1: Cohort Construction
This notebook constructs the ARF NIV study cohort from CLIF tables.

## Steps
1. Construct hospital blocks.
2. Identify ED encounters.
3. Apply inclusion and exclusion criteria.

### Set up

In [0]:
# --- Load required dependenies ---
import yaml
import pathlib
import pandas as pd
from datetime import datetime

# --- Load global and local configuration ---
# Project root
project_root = pathlib.Path.cwd()

# Load configuration files
while not (project_root / "config").is_dir():
  if project_root.parent == project_root:
    raise Exception("Configuration folder not found")
  project_root = project_root.parent

with open(project_root / "config" / "config.yaml", encoding = "utf-8") as f:
  config = yaml.safe_load(f)

with open(project_root / "config" / "global_config.yaml", encoding = "utf-8") as f:
  global_config = yaml.safe_load(f)

# Assign global config values
config_sql = {key: value for key, value in global_config.items() if key != "required_tables"}

# Assign local config values
tables_location = config["tables_location"]
project_location = config["project_location"]
institution = config["institution"]
time_zone = config["time_zone"]
file_type = config["file_type"]

# --- Loading project related clinical thresholds ---
while not (project_root / "outlier-thresholds").is_dir():
  if project_root.parent == project_root:
    raise Exception("Threshold folder not found")
  project_root = project_root.parent

thresholds_path = project_root / "outlier-thresholds" / "project_outlier_thresholds.csv"
thresholds = pd.read_csv(thresholds_path).set_index("variable_name")

# Add more thresholds as needed
config_sql["age_min"] = float(thresholds.loc["age_at_admission", "lower_limit"])
config_sql["age_max"] = float(thresholds.loc["age_at_admission", "upper_limit"])

# --- Expose configuration to SQL ---
spark.createDataFrame([config_sql]).createOrReplaceTempView("config_sql")

# --- Load CLIF tables ---
# Check if tables exist
clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(tables_location) if item.isDir()}

required_tables = set(global_config["required_tables"])

missing_table = sorted(required_tables - clif_table_filenames)

if len(missing_table) > 0:
  raise Exception(f"Missing tables: {missing_table}")
else: 
  print("All tables present.")

# Register source CLIF tables as temp views
for table_name in required_tables:
  source = spark.read.parquet(f"{tables_location}/{table_name}")
  source.createOrReplaceTempView(f"src_{table_name}")
  print(f"Registered temp view for {table_name}")

In [0]:
%sql
-- Create column that stores the time of previous discharge
-- Create column that stores the gap in hours between current admission and prior discharge
-- Flag a hospitalization if it represents a new hospital block
-- Compute cumulative block number per patient (e.g., first hospital block =1 , second = 2, etc.)
-- Determine end of hospitalization time for discharge location
create or replace temp view working_hospitalization as
with step1 as (
  select
    h.*,
    c.hospital_block_time_hrs as hospital_block_time_hrs,
    lag(h.discharge_dttm, 1) over (
        partition by h.patient_id
        order by h.admission_dttm
      ) as prev_discharge_dttm
  from
    src_clif_hospitalization as h cross join config_sql as c
  where
    h.age_at_admission >= c.age_min
    and h.admission_dttm >= cast(c.study_start as timestamp)
),
step2 as (
  select
    *,
    case
      when
        prev_discharge_dttm is null
        or cast(admission_dttm as timestamp) - cast(prev_discharge_dttm as timestamp)
          > hospital_block_time_hrs * interval '1' hour
      then
        1
      else 0
    end as new_block
  from
    step1
)
select
  * except (hospital_block_time_hrs, prev_discharge_dttm, new_block),
  sum(coalesce(new_block, 1)) over (
      partition by patient_id
      order by admission_dttm
      rows between unbounded preceding and current row
    ) as block_num,
  coalesce(discharge_dttm, admission_dttm) as end_ts
from
  step2
order by
  patient_id,
  admission_dttm;

In [0]:
%sql
-- Stores information on grouped hospitalizations (hospital blocks)
-- Stores the patient age on first admission within block
-- Stores the start of the hospital block time
-- Stores the end of the hospital block time
-- Stores the discharge location from last admission within block
-- Stores the number of hospitalizations within block
create or replace temp view hospital_blocks as
with step1 as (
  select
    *,
    row_number() over (
        partition by patient_id, block_num
        order by end_ts desc NULLS last, hospitalization_id asc
      ) as index_by_descending_end_ts
  from
    working_hospitalization
)
select
  s.patient_id,
  s.block_num,
  min(s.age_at_admission) as age_at_admission,
  min(s.admission_dttm) as block_start_admit,
  max(s.discharge_dttm) as block_end_discharge,
  count(*) as num_hospitalizations,
  max(
    case
      when s.index_by_descending_end_ts = 1 then s.discharge_category
      else NULL
    end
  ) as discharge_location,
  concat(s.patient_id, '_', cast(min(s.admission_dttm) as string)) as hospital_block_id
from
  step1 as s cross join config_sql as c
group by
  s.patient_id,
  s.block_num
having
  min(s.admission_dttm) <= min(c.study_end)
  and min(s.age_at_admission) <= min(c.age_max);